# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Reusable Functions

In [0]:
def parse_mixed_date(df, col_name, output_col=None):
    out_col = output_col if output_col else col_name

    return df.withColumn(
        out_col,
        when(
            trim(col(col_name)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(trim(col(col_name)), "dd-MM-yyyy")
        ).when(
            trim(col(col_name)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(trim(col(col_name)),"dd/MM/yyyy")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy/MM/dd")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}-\d{1,2}-\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy-MM-dd")
        ).otherwise(None)
    )

# Date Cleaning

## fx_rate Table

In [0]:
fx_rate_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_fx_rate`""")

### TypeCasting

In [0]:
fx_rate_df=fx_rate_df.withColumn("fx_rate_to_gbp",col("fx_rate_to_gbp").cast(DoubleType()))

In [0]:
fx_rate_df=parse_mixed_date(fx_rate_df,"effective_date")
fx_rate_df=fx_rate_df.withColumn("effective_date",col("effective_date").cast(DateType()))

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {SILVER_SCHEMA_PATH}""")

# Saving Dataframe

In [0]:
fx_rate_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_fx_rate_df`")